<!-- ============================================================ -->
<!-- NOTEBOOK HEADER — MLOps Introductory Course on GCP           -->
<!-- ============================================================ -->

<div style="border-bottom: 3px solid #4285F4; padding-bottom: 12px; margin-bottom: 20px;">

<div style="display: flex; align-items: center; justify-content: space-between;">
  <div>
    <img src="https://www.isae-supaero.fr/wp-content/uploads/2025/03/logo.svg" width="180">
  </div>
  <div style="text-align: right;">
    <img src="https://user-images.githubusercontent.com/63151412/167391313-4683cc69-2bf6-4597-b767-5c18e2bbbfa0.png" width="180">
  </div>
</div>

# Lab 04b — Production Pipeline Patterns

**Course:** MLOps Introductory Course on GCP · M2 Data Science · ISAE-SUPAERO  
**Lab created by:** Headmind Partners AI & Blockchain  
**Estimated duration:** ~1h15

</div>

## 📋 Lab Overview

In Lab 04a you built a basic three-step pipeline. Production ML systems need more: they must **validate data** before training, **evaluate models against quality gates**, and **conditionally deploy** only if the model is good enough. These patterns prevent bad models from reaching production and make your ML system self-healing.

### Learning Objectives

1. Build a **data validation component** that acts as a quality gate.
2. Use **`dsl.If`** to add conditional branching to a pipeline.
3. Implement a **model quality gate** that blocks deployment of underperforming models.
4. Register a model to the **Vertex AI Model Registry** from within a pipeline component.
5. Run the full production pipeline and observe how conditions control execution flow.
6. Compare pipeline runs with different parameters and understand the impact on branching.

### Notebook Structure

| # | Section | Focus |
|---|---------|-------|
| 0 | Setup | Imports, GCP configuration, reuse components from Lab 04a |
| 1 | Data Validation | Build a component that checks data quality |
| 2 | Model Quality Gate | Modify evaluation to return a pass/fail signal |
| 3 | Conditional Pipeline | Use `dsl.If` for branching logic |
| 4 | Model Registration | Register a passing model to Vertex AI Model Registry |
| 5 | Run & Compare | Execute the pipeline and observe conditional behavior |
| 6 | Cleanup | Delete pipeline runs, models, and artifacts |

### How to Read This Notebook

- **`# TODO`** — Code you need to write. Look for the `######` delimiters.
- **`✏️ Question`** — A conceptual question. Write your answer in the markdown cell below it.
- Cells **without** a TODO are provided — read them, run them, and make sure you understand them.
- Documentation links are provided in 📖 callouts whenever a new API is introduced.

---
## 0 · Setup

### 0.1 Install dependencies

In [ ]:
%pip install --upgrade --quiet kfp google-cloud-aiplatform google-cloud-pipeline-components

### 0.2 Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

from kfp import dsl
from kfp.dsl import Input, Output, Dataset, Model, Metrics
from kfp import compiler

import google.cloud.aiplatform as aiplatform

import kfp
print(f"KFP SDK version: {kfp.__version__}")
print(f"Vertex AI SDK version: {aiplatform.__version__}")

KFP SDK version: 2.16.0
Vertex AI SDK version: 1.140.0


### 0.3 Configuration

Use the same project settings as Lab 04a.

In [ ]:
# ── Constants ──
PROJECT_ID = "isae-sdd-481407"         # @param {type:"string"}
LOCATION = "europe-west3"               # @param {type:"string"}
BUCKET_URI = "gs://bucket-lab04"     # @param {type:"string"}

##############################  TODO  ##############################
# Set your_name to a unique lowercase identifier (e.g. first letter of first name + last name).
your_name = "mregaieg"  # TODO
####################################################################

# Pipeline artifacts will be stored under this root
PIPELINE_ROOT = f"{BUCKET_URI}/pipeline-root/lab04b/{your_name}"

aiplatform.init(
    project=PROJECT_ID,
    location=LOCATION,
    staging_bucket=BUCKET_URI,
)
print(f"✅ Vertex AI initialized — project={PROJECT_ID}, location={LOCATION}")

✅ Vertex AI initialized — project=isae-sdd-481407, location=europe-west3


In [ ]:
PIPELINE_DISPLAY_NAME = f"lab04b-production-{your_name}"
print(f"Pipeline display name: {PIPELINE_DISPLAY_NAME}")

Pipeline display name: lab04b-production-mregaieg


### 0.4 Reuse components from Lab 04a

We reuse the `load_and_split_data` and `train_model` components from Lab 04a. They are redefined here for convenience.

In [ ]:
# Reused from Lab 04a — data preparation component
@dsl.component(
    base_image="python:3.10",
    packages_to_install=["scikit-learn==1.5.2", "pandas==2.2.3"],
)
def load_and_split_data(
    test_size: float,
    random_state: int,
    n_samples: int,
    train_dataset: Output[Dataset],
    test_dataset: Output[Dataset],
):
    """Fetch the Covertype dataset, subsample, and split into train/test."""
    import pandas as pd
    from sklearn.datasets import fetch_covtype
    from sklearn.model_selection import train_test_split

    data = fetch_covtype(as_frame=True)
    df = data.frame
    df = df.sample(n=min(n_samples, len(df)), random_state=random_state)
    train_df, test_df = train_test_split(
        df, test_size=test_size, random_state=random_state, stratify=df['Cover_Type']
    )
    train_df.to_csv(train_dataset.path, index=False)
    test_df.to_csv(test_dataset.path, index=False)
    print(f"✅ Data loaded: {len(train_df)} train / {len(test_df)} test samples")

print("✅ load_and_split_data component defined.")

✅ load_and_split_data component defined.


In [ ]:
# Reused from Lab 04a — training component
@dsl.component(
    base_image="python:3.10",
    packages_to_install=["scikit-learn==1.5.2", "pandas==2.2.3"],
)
def train_model(
    train_dataset: Input[Dataset],
    n_estimators: int,
    max_depth: int,
    random_state: int,
    model_artifact: Output[Model],
):
    """Train a RandomForestClassifier on the training set."""
    import os
    import pickle
    import pandas as pd
    from sklearn.ensemble import RandomForestClassifier

    train_df = pd.read_csv(train_dataset.path)
    X_train = train_df.drop(columns=["Cover_Type"])
    y_train = train_df["Cover_Type"]
    clf = RandomForestClassifier(
        n_estimators=n_estimators, max_depth=max_depth, random_state=random_state
    )
    clf.fit(X_train, y_train)

    os.makedirs(model_artifact.path, exist_ok=True)
    with open(os.path.join(model_artifact.path, "model.pkl"), "wb") as f:
        pickle.dump(clf, f)

    model_artifact.metadata["n_estimators"] = n_estimators
    model_artifact.metadata["max_depth"] = max_depth
    model_artifact.metadata["framework"] = "scikit-learn"
    print(f"✅ Model trained: n_estimators={n_estimators}, max_depth={max_depth}")

print("✅ train_model component defined.")

✅ train_model component defined.


---
## 1 · Data Validation Component

In production, you should never train a model on data you haven't validated. Data can drift over time, contain unexpected nulls, or have schema violations. A **data validation component** acts as a quality gate: it checks the data and signals whether training should proceed.

Unlike previous components we worked on, this one will return a value. We need to match the type annotations accordingly.

> 📖 **Docs:** [KFP component return values](https://www.kubeflow.org/docs/components/pipelines/user-guides/components/lightweight-python-components/#python-function-based-components) — a component can return simple types (`str`, `int`, `float`, `bool`) that downstream tasks can use.

In [ ]:
##############################  TODO  ##############################
# Build a data validation component that checks:
# 1. No missing values (NaN) in the dataset
# 2. The target column 'Cover_Type' exists
# 3. Cover_Type values are in the expected range [1, 7]
# 4. The dataset has at least min_rows rows
#
# The component should return a string: 'passed' if all checks pass,
# 'failed' otherwise. It should also print which checks pass/fail.

@dsl.component(
    base_image="python:3.10",
    packages_to_install=["pandas==2.2.3"],
)
def validate_data(
    dataset: Input[Dataset],
    min_rows: int,
) -> str:
    """Validate data quality. Returns "passed" or "failed"."""
    import pandas as pd

    df = pd.read_csv(dataset.path)
    checks_passed = True

    # Check 1: No missing values
    if df.isna().values.any():
      checks_passed=False  # TODO

    # Check 2: Target column exists
    if not("Cover_Type" in df.columns):
      checks_passed=False# TODO

    # Check 3: Cover_Type values in range [1, 7]
    if "Cover_Type" in df.columns and not (df["Cover_Type"].between(1, 7).all()):
       checks_passed=False# TODO

    # Check 4: Minimum number of rows
    if len(df)<min_rows :
      checks_passed = False# TODO

    status = "passed" if checks_passed else "failed"
    print(f"\n{'='*40}")
    print(f"Validation result: {status.upper()}")
    print(f"{'='*40}")
    return status
####################################################################

print("✅ validate_data component defined.")

✅ validate_data component defined.


**✏️ Question 1 — Data validation in production**

a) In this lab we validate a static dataset. In a real production system, what kind of data quality issues might appear over time that wouldn't be caught by a simple schema check?

b) How would you extend this component to detect **data drift** (i.e., the distribution of incoming data has shifted compared to the training data)?

---
*Your answer:*

**a)** In this lab, the dataset is fixed, but in real life the data is often online and keeps changing. So even if the schema is still correct, the data can become different from before. For example, some values may appear more often, the average of a feature may change, or the data may become noisier. These problems would not be caught by a simple schema check because the columns would still exist and have the right type.

**b)** A good extension would be to add another component for data drift detection. This component would compare the new data with the training data and check if the distributions are still similar. For example, it could compare the mean, standard deviation, or the frequency of some values. If the difference is too large, it could raise a warning that the data has changed.

---

---
## 2 · Evaluation with Quality Gate

In Lab 04a, the evaluation component simply logged metrics. In a production pipeline, evaluation must also **make a decision**: is this model good enough to deploy? We extend the component to return both metrics and a pass/fail signal.

> 📖 **Docs:** [KFP `NamedTuple` outputs](https://www.kubeflow.org/docs/components/pipelines/user-guides/data-handling/parameters/) — use `NamedTuple` to return multiple typed values from a component.

In [ ]:
##############################  TODO  ##############################
# Complete the evaluate_with_gate component.
# It should:
# 1. Load test data and the trained model
# 2. Compute accuracy and weighted F1
# 3. Log metrics to the Metrics artifact
# 4. Return "passed" if accuracy >= threshold, "failed" otherwise
#
# The return type is a NamedTuple with fields:
#   - accuracy (float), f1 (float), decision (str: "passed"/"failed")

from typing import NamedTuple


@dsl.component(
    base_image="python:3.10",
    packages_to_install=["scikit-learn==1.5.2", "pandas==2.2.3"],
)
def evaluate_with_gate(
    test_dataset: Input[Dataset],
    model_artifact: Input[Model],
    accuracy_threshold: float,
    metrics: Output[Metrics],
) -> NamedTuple('eval_outputs', accuracy=float, f1=float, decision=str):
    """Evaluate model and return a pass/fail decision."""
    import os
    import pickle
    import pandas as pd
    from sklearn.metrics import accuracy_score, f1_score
    from collections import namedtuple
    from typing import NamedTuple

    test_df = pd.read_csv(test_dataset.path)
    X_test = test_df.drop(columns=["Cover_Type"])
    y_test = test_df["Cover_Type"]

    with open(os.path.join(model_artifact.path, "model.pkl"), "rb") as f:
        clf = pickle.load(f)

    y_pred = clf.predict(X_test)  # TODO — predict

    accuracy = accuracy_score(y_pred,y_test)  # TODO — compute accuracy
    f1 = f1_score(y_test, y_pred, average="weighted")        # TODO — compute weighted F1

    # TODO — log metrics
    metrics.log_metric("accuracy",accuracy)  # TODO
    metrics.log_metric("f1",f1)  # TODO

    # TODO — make decision based on threshold
    decision = "passed" if accuracy >  accuracy_threshold else "failed" # TODO

    print(f"Accuracy: {accuracy:.4f} (threshold: {accuracy_threshold})")
    print(f"Decision: {decision.upper()}")

    eval_outputs = NamedTuple('eval_outputs', accuracy=float, f1=float, decision=str)

    return eval_outputs(round(accuracy, 4), round(f1, 4), decision)
####################################################################

print("✅ evaluate_with_gate component defined.")

✅ evaluate_with_gate component defined.


---
## 3 · Conditional Pipeline with `dsl.If`

A production pipeline should not blindly execute every step. **Conditional logic** lets you skip expensive or dangerous steps (like deployment) when preconditions are not met.

KFP provides `dsl.If` (and `dsl.Else`) to branch pipeline execution based on the output of a previous component.

> 📖 **Docs:** [KFP control flow — `dsl.If`](https://www.kubeflow.org/docs/components/pipelines/user-guides/core-functions/control-flow/)

### 3.1 Model registration component

This component registers the model to the **Vertex AI Model Registry** — the same registry you used in Lab 02b. This is **provided** so you can focus on the pipeline logic.

In [ ]:
# Model registration component (provided)
@dsl.component(
    base_image="python:3.10",
    packages_to_install=["google-cloud-aiplatform==1.74.0"],
)
def register_model(
    model_artifact: Input[Model],
    project: str,
    location: str,
    display_name: str,
) -> str:
    """Register the model in Vertex AI Model Registry."""
    from google.cloud import aiplatform

    aiplatform.init(project=project, location=location)

    # Upload as an unmanaged model (no serving container needed)
    model = aiplatform.Model.upload(
        display_name=display_name,
        artifact_uri=model_artifact.uri,
        serving_container_image_uri="us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-3:latest",
    )

    print(f"✅ Model registered: {model.display_name}")
    print(f"   Resource name: {model.resource_name}")
    return model.resource_name

print("✅ register_model component defined.")

✅ register_model component defined.


### 3.2 Notification component

In production, you'd want to notify the team when a model fails the quality gate. Here we simulate this with a simple logging component.

In [ ]:
# Notification component (provided)
@dsl.component(
    base_image="python:3.10",
)
def notify_failure(
    accuracy: float,
    threshold: float,
):
    """Send a notification when model fails quality gate."""
    print(f"⚠️ MODEL QUALITY GATE FAILED")
    print(f"   Accuracy: {accuracy:.4f}")
    print(f"   Threshold: {threshold}")
    print(f"   The model was NOT registered.")
    print(f"   In production, this would trigger an alert to the ML team.")

print("✅ notify_failure component defined.")

✅ notify_failure component defined.


### 3.3 Define the production pipeline

Now compose everything into a pipeline with conditional logic:

1. **Load & split** data
2. **Validate** the training data → if validation fails, stop
3. **Train** the model (only if data is valid)
4. **Evaluate** with quality gate
5. **If** evaluation passes → **register** the model
6. **Else** → **notify** the team

In [ ]:
##############################  TODO  ##############################
# Complete the production pipeline.
# The structure should be:
#   1. data_task = ...
#   2. validate_task = ...
#   3. with dsl.If(...):  # check if validation passed
#        a. train_task = ...
#        b. eval_task = ...
#        c. with dsl.If(...):
#              ...
#           with dsl.Else:
#              ...
#
# Hints:
# - validate_task.output gives the string return value
# - eval_task.outputs["decision"] gives the named output
# - eval_task.outputs["accuracy"] gives the accuracy value

@dsl.pipeline(
    name="covertype-production-pipeline-mohamed",
    description="Production pipeline with data validation and model quality gate.",
)
def production_pipeline(
    n_estimators: int = 100,
    max_depth: int = 10,
    test_size: float = 0.2,
    n_samples: int = 20000,
    random_state: int = 42,
    accuracy_threshold: float = 0.80,
    min_rows: int = 1000,
    model_display_name: str = "covertype-rf",
):

    # Step 1: Load and split data
    data_task = load_and_split_data(
        test_size=test_size,
        random_state=random_state,
        n_samples=n_samples,
    )

    # Step 2: Validate training data
    validate_task = validate_data(
    dataset=data_task.outputs["train_dataset"],
    min_rows=min_rows,
)  # TODO

    # Step 3: Conditional — only train if data is valid
    with dsl.If(validate_task.output=='passed'):  # TODO — use dsl.If

        # Step 3a: Train model
        train_task = train_model(
        train_dataset=data_task.outputs["train_dataset"],
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=random_state,
    )  # TODO

        # Step 3b: Evaluate with quality gate
        eval_task =  evaluate_with_gate(
        test_dataset=data_task.outputs["test_dataset"],
        model_artifact=train_task.outputs["model_artifact"],
        accuracy_threshold=accuracy_threshold,
    )   # TODO

        # Step 3c: Conditional — register or notify
        with dsl.If(eval_task.outputs['decision']=="passed"):
          register_model(
            model_artifact=train_task.outputs["model_artifact"],
            project=PROJECT_ID,
            location=LOCATION,
            display_name=model_display_name,

        )
        with dsl.Else():
          notify_failure(
             accuracy=eval_task.outputs["accuracy"],
             threshold=accuracy_threshold,
)# TODO — use dsl.If / dsl.Else
####################################################################

print("✅ Production pipeline defined.")

✅ Production pipeline defined.


**✏️ Question 2 — Conditional pipelines**

a) What would happen if we removed the `dsl.If` around the training step and always trained, even with invalid data? What downstream effects could this have?

b) In the current pipeline, if data validation fails, the entire training/evaluation/registration branch is skipped. Can you think of a scenario where you'd want to **still train** on partially invalid data but with an extra preprocessing step?

---
*Your answer:*

**a)** If the `dsl.If` was removed, the pipeline would train the model even when the data is not valid. This could create a bad model because the training data may contain errors, missing values, or wrong target values. Then the evaluation would also be less trustworthy, and the model could even be registered by mistake.

**b)** Yes, this can happen when the data is not completely wrong but just needs some cleaning. For example, if a few values are missing, a preprocessing step could fill them or remove a small number of bad rows. In that case, it can still make sense to train the model after this extra preprocessing step instead of stopping the whole pipeline.


---

---
## 4 · Compile & Run the Production Pipeline

### 4.1 Compile

In [ ]:
# Compile the production pipeline
compiler.Compiler().compile(
    pipeline_func=production_pipeline,
    package_path="production_pipeline.yaml",
)
print("✅ Production pipeline compiled to production_pipeline.yaml")

✅ Production pipeline compiled to production_pipeline.yaml


### 4.2 Run with passing thresholds

First, run the pipeline with a reasonable accuracy threshold that the model should pass.

In [ ]:
##############################  TODO  ##############################
# Submit the production pipeline with parameters that should
# result in the model PASSING the quality gate.
# Hint: try to remember the metric results from lab04a

job_pass = aiplatform.PipelineJob(
    display_name=f"{PIPELINE_DISPLAY_NAME}-pass",
    template_path="production_pipeline.yaml",
    pipeline_root=PIPELINE_ROOT,
    parameter_values={
        "accuracy_threshold": 0.7,      # TODO
        "model_display_name": f"covertype-rf-{your_name}",
    },
)
job_pass.run()  # TODO — run the job
####################################################################

print(f"✅ Pipeline completed: {job_pass.display_name}")

✅ Pipeline completed: lab04b-production-mregaieg-pass


**✏️ Question 3 — Pipeline execution flow**

a) Look at the pipeline DAG in the Vertex AI console. Which components were executed, and which were skipped? Does this match your expectations?

b) Where was the model registered? Go to the Vertex AI Model Registry in the console and verify.

---
*Your answer:*

**a)** From the pipeline DAG, the components **load-and-split-data**, **validate-data**, **train-model**, and **evaluate-with-gate** were executed successfully. Since the evaluation decision was **"passed"**, the **register-model** component was also executed. The **notify-failure** component was skipped because it is only triggered when the model does not meet the accuracy threshold. This behavior matches the expected logic of the pipeline.

**b)** The model was registered in the **Vertex AI Model Registry** of the same project and region used in the pipeline ( project `isae-sdd-481407`, region `europe-west3`). In the Model Registry, the model appears with the display name defined in the pipeline, which in this case is **"covertype-rf"**.

---

---
## 5 · Test the Failure Path

To verify that your quality gates work correctly, run the pipeline with an **unrealistically high** accuracy threshold that the model cannot meet.

In [17]:
##############################  TODO  ##############################
# Submit a second run with another accuracy_threshold.
# The model should FAIL the quality gate, triggering notify_failure
# instead of register_model.

job_fail = aiplatform.PipelineJob(
    display_name=f"{PIPELINE_DISPLAY_NAME}-fail",
    template_path="production_pipeline.yaml",
    pipeline_root=PIPELINE_ROOT,
    parameter_values={
        "accuracy_threshold": 0.9,  # TODO
    },
)
job_fail.run(sync=False)
####################################################################

print(f"✅ Pipeline submitted")

✅ Pipeline submitted


In [18]:
job_fail.wait()
print(f"✅ Pipeline completed — state: {job_fail.state.name}")

✅ Pipeline completed — state: PIPELINE_STATE_SUCCEEDED


### 5.1 Compare both runs

Compare the two runs side by side. Pay attention to which components were executed vs. skipped in each run.

In [19]:
# List and compare pipeline runs
for j in [job_pass, job_fail]:
    print(f"Run: {j.display_name}")
    print(f"  State: {j.state.name}")
    params = dict(j.gca_resource.runtime_config.parameter_values)
    print(f"  Accuracy threshold: {params.get('accuracy_threshold', 'default')}")
    print(f"  Tasks:")
    tasks = sorted(j.gca_resource.job_detail.task_details, key=lambda t: t.start_time or t.create_time)
    for task in tasks:
        print(f"    {task.task_name:<30} {task.state.name}")
    print()

Run: lab04b-production-mregaieg-pass
  State: PIPELINE_STATE_SUCCEEDED
  Accuracy threshold: 0.7
  Tasks:
    covertype-production-pipeline-mohamed-20260310155100 SUCCEEDED
    load-and-split-data            SUCCEEDED
    validate-data                  SUCCEEDED
    condition-1                    SUCCEEDED
    train-model                    SUCCEEDED
    evaluate-with-gate             SUCCEEDED
    condition-branches-2           SUCCEEDED
    condition-4                    NOT_TRIGGERED
    condition-3                    SUCCEEDED
    register-model                 SUCCEEDED

Run: lab04b-production-mregaieg-fail
  State: PIPELINE_STATE_SUCCEEDED
  Accuracy threshold: 0.9
  Tasks:
    covertype-production-pipeline-mohamed-20260310160246 SUCCEEDED
    load-and-split-data            SKIPPED
    validate-data                  SKIPPED
    condition-1                    SUCCEEDED
    train-model                    SKIPPED
    evaluate-with-gate             SUCCEEDED
    condition-branches-2 

**✏️ Question 4 — Production safeguards**

a) In the second run, the model trained but was not registered because it failed the accuracy gate. Is there wasted compute in this scenario? How could you reduce it?

b) You are asked to add a **rollback mechanism**: if a newly deployed model performs worse than the previous version in production, automatically revert. How would you design this as part of the pipeline?

---
*Your answer:*

**a)** Yes, there is some wasted compute because the model is trained even if it is not registered in the end. Training can take time and resources, so if the model fails the accuracy gate, that work does not directly produce a usable model. To reduce this, the pipeline could add an earlier check before full training, or use a smaller and faster test run first.

**b)** A rollback mechanism could keep the previous model as the current production version and only send a small part of the traffic to the new model. Then the system could monitor the new model’s performance. If it performs worse than the old one, the traffic could be switched back to the previous model automatically. This would make deployment safer.


---

---
## 6 · Cleanup

Clean up all resources created during this lab.

In [20]:
# Delete registered models
models = aiplatform.Model.list(
    filter=f'display_name="covertype-rf-{your_name}"'
)
for m in models:
    try:
        m.delete()
        print(f"✅ Deleted model: {m.display_name}")
    except Exception as e:
        print(f"⚠️ Could not delete {m.display_name}: {e}")

✅ Deleted model: covertype-rf-mregaieg


In [21]:
# Delete pipeline jobs
for j in [job_pass, job_fail]:
    try:
        j.delete()
        print(f"✅ Deleted pipeline job: {j.display_name}")
    except Exception as e:
        print(f"⚠️ Could not delete {j.display_name}: {e}")

✅ Deleted pipeline job: lab04b-production-mregaieg-pass
✅ Deleted pipeline job: lab04b-production-mregaieg-fail


---
## Summary

In this lab, you learned to:

| Step | What you did | Tool / Feature used |
|------|-------------|---------------------|
| Data Validation | Built a custom quality gate component | `@dsl.component`, schema checks |
| Quality Gate | Extended evaluation with pass/fail decision | `NamedTuple` returns |
| Conditional Logic | Controlled pipeline flow with branching | `dsl.If`, `dsl.Else` |
| Model Registration | Registered passing models to the registry | `aiplatform.Model.upload()` |
| Failure Path | Tested that quality gates block bad models | `notify_failure` component |

**Key takeaway:** Production ML pipelines are not just about training models — they are about building **automated safeguards** that ensure only validated data produces only quality-approved models. The conditional patterns you learned here (`dsl.If`, quality gates, validation components) are the building blocks of a self-healing ML system.

**Next lab:** In Lab 05, you will build a **RAG chatbot** using LangChain and Vertex AI, bringing together the deployment skills from previous labs with the latest in generative AI.